# 51 — Cấu trúc dữ liệu và kiểu dữ liệu trong pandas 3.0

Notebook đầu của chuỗi pandas. Nó không dạy lại `pd.DataFrame({"a": [1, 2, 3]})`
— nó mổ một frame **dữ liệu thị trường thật** lấy từ `finlens` và trả lời ba
câu mà pandas 3.0 vừa đổi câu trả lời:

1. Một `DataFrame` **thực ra là gì** — và `Index` làm gì trong đó
2. ⚠️ **`str` đã thay `object` làm kiểu mặc định cho chuỗi**
3. `pd.NA` khác `np.nan` thế nào, và vì sao `NA > 0` không trả về `False`

Cộng thêm phần đo bộ nhớ — chỗ một lựa chọn kiểu dữ liệu tiết kiệm được **40
lần**.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

print("pandas", pd.__version__, "· numpy", np.__version__, "· finlens", finlens.build_info()["version"])

pandas 3.0.5 · numpy 2.5.2 · finlens 1.3.0


## 1 · Ba khối xây nên mọi thứ

`Series` là một cột. `DataFrame` là nhiều cột dùng chung một `Index`. `Index`
là nhãn hàng — và nó **không phải** số thứ tự, dù mặc định trông giống thế.

In [2]:
gia = client.eod.stock.ohlcv(["HPG", "VCB", "FPT"], start=lui_ngay(HOM_NAY, thang=3))

print(f"type: {type(gia).__name__}")
print(f"shape: {gia.shape}  →  {gia.shape[0]} hàng × {gia.shape[1]} cột")
print(f"index: {type(gia.index).__name__}  {gia.index[:3].tolist()} …")
print(f"columns: {type(gia.columns).__name__}")
gia.head(3)

type: DataFrame
shape: (195, 7)  →  195 hàng × 7 cột
index: RangeIndex  [0, 1, 2] …
columns: Index


,symbol,date,open,high,low,close,volume
0,FPT,2026-05-14,70.13,74.28,70.04,72.90,22881586.0
1,FPT,2026-05-15,73.19,73.19,71.71,71.91,6386100.0
2,FPT,2026-05-18,71.91,74.28,71.02,73.88,19101114.0


In [3]:
mot_cot = gia["close"]

print(f"gia['close'] là  : {type(mot_cot).__name__}")
print(f"gia[['close']] là: {type(gia[['close']]).__name__}   ← hai dấu ngoặc = DataFrame")
print(f"\nSeries có tên riêng: {mot_cot.name!r}")
print(f"và dùng chung index với frame mẹ: {mot_cot.index.equals(gia.index)}")

gia['close'] là  : Series
gia[['close']] là: DataFrame   ← hai dấu ngoặc = DataFrame

Series có tên riêng: 'close'
và dùng chung index với frame mẹ: True


### `Index` không phải số thứ tự

Đây là chỗ người mới hay ngã. Sau khi lọc, nhãn hàng **giữ nguyên giá trị cũ** —
nó không được đánh số lại.

In [4]:
hpg = gia[gia["symbol"] == "HPG"]

print(f"5 nhãn đầu của frame gốc: {gia.index[:5].tolist()}")
print(f"5 nhãn đầu sau khi lọc  : {hpg.index[:5].tolist()}   ← không phải 0,1,2,3,4")
print(f"\nhpg.iloc[0] lấy hàng ĐẦU TIÊN theo vị trí   → close = {hpg.iloc[0]['close']}")
print(f"hpg.loc[{hpg.index[0]}] lấy hàng có NHÃN đó  → close = {hpg.loc[hpg.index[0]]['close']}")

5 nhãn đầu của frame gốc: [0, 1, 2, 3, 4]
5 nhãn đầu sau khi lọc  : [65, 66, 67, 68, 69]   ← không phải 0,1,2,3,4

hpg.iloc[0] lấy hàng ĐẦU TIÊN theo vị trí   → close = 24.59
hpg.loc[65] lấy hàng có NHÃN đó  → close = 24.59


Khi frame gốc bắt đầu từ nhãn 0 thì `iloc[0]` và `loc[0]` trùng nhau, nên lỗi
ẩn mình. Sau một lần lọc là chúng tách ra ngay.

`reset_index(drop=True)` đánh số lại; bỏ `drop=True` thì nhãn cũ thành một cột.

In [5]:
print("reset_index(drop=True) :", hpg.reset_index(drop=True).index[:5].tolist())
print("reset_index()          :", hpg.reset_index().columns.tolist())

reset_index(drop=True) : [0, 1, 2, 3, 4]
reset_index()          : ['index', 'symbol', 'date', 'open', 'high', 'low', 'close', 'volume']


## 2 · ⚠️ pandas 3.0: `str` đã thay `object`

Đây là thay đổi lớn nhất của bản 3.0 với người làm dữ liệu thị trường, vì cột
mã chứng khoán có mặt ở mọi frame.

In [6]:
print("dtypes của frame giá:")
print(gia.dtypes.to_string())

print(f"\ndtype của cột symbol: {gia['symbol'].dtype}  ({type(gia['symbol'].dtype).__name__})")
print(f"Ở pandas 2.x cột này là: object")

dtypes của frame giá:
symbol            string
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume           float64

dtype của cột symbol: string  (StringDtype)
Ở pandas 2.x cột này là: object


In [7]:
# Kiểm chứng: một Series chuỗi tạo mới cũng ra `str`, không còn `object`
print(f"pd.Series(['HPG','VCB']).dtype = {pd.Series(['HPG', 'VCB']).dtype}")
print(f"pd.options.future.infer_string = {pd.options.future.infer_string}")

pd.Series(['HPG','VCB']).dtype = str
pd.options.future.infer_string = True


Vì sao đổi: `object` là một mảng **con trỏ tới object Python**. Mỗi ô là một
`str` riêng nằm rải rác trên heap, nên pandas không tối ưu được gì cả và kiểu
dữ liệu không nói lên nội dung — `object` có thể chứa chuỗi, list, hay bất cứ
thứ gì.

`str` là kiểu **có ý nghĩa**: pandas biết chắc mọi ô là chuỗi hoặc thiếu.

In [8]:
tron = pd.Series(["HPG", 123, [1, 2]])
print(f"Series trộn nhiều loại → dtype {tron.dtype}   ← object vẫn tồn tại, chỉ không còn mặc định")

Series trộn nhiều loại → dtype object   ← object vẫn tồn tại, chỉ không còn mặc định


### `str` có hai kiểu lưu trữ

Mặc định là `python`. Cài thêm `pyarrow` thì pandas dùng nó và tiết kiệm bộ nhớ
đáng kể — nhưng đó là lựa chọn của bạn, không phải mặc định.

In [9]:
print(f"storage hiện tại: {gia['symbol'].dtype.storage!r}")
try:
    import pyarrow

    print(f"pyarrow {pyarrow.__version__} đã cài")
except ImportError:
    print("pyarrow chưa cài → dùng storage 'python'. `pip install pyarrow` để đổi.")

storage hiện tại: 'python'
pyarrow chưa cài → dùng storage 'python'. `pip install pyarrow` để đổi.


### ⚠️ Có **hai** kiểu chuỗi, và bạn vừa nhìn thấy cả hai

Để ý hai dòng in ở trên: frame finlens báo `string`, còn `pd.Series(["HPG"])`
báo `str`. Đó không phải cách viết tắt khác nhau của cùng một thứ — chúng là
hai `StringDtype` với **giá trị thiếu khác nhau**.

In [10]:
dt_finlens = gia["symbol"].dtype
dt_pandas = pd.Series(["HPG", "VCB"]).dtype

print(f"{'nguồn':<22}{'repr':<10}{'storage':<12}{'na_value'}")
for ten, d in [("finlens trả về", dt_finlens), ("pandas mặc định", dt_pandas)]:
    print(f"{ten:<22}{str(d):<10}{d.storage!r:<12}{d.na_value!r}")

print()
print(f"Hai kiểu này bằng nhau? {dt_finlens == dt_pandas}")

nguồn                 repr      storage     na_value
finlens trả về        string    'python'    <NA>
pandas mặc định       str       'python'    nan

Hai kiểu này bằng nhau? False


Hệ quả nằm ở **kiểu của boolean mask** — thứ bạn dùng để lọc mọi lúc:

In [11]:
co_thieu_na = pd.Series(["HPG", None], dtype=dt_finlens)
co_thieu_nan = pd.Series(["HPG", None], dtype=dt_pandas)

print(f"dtype 'string' → ô thiếu là {co_thieu_na[1]!r}")
print(f"  mask (== 'HPG'): {(co_thieu_na == 'HPG').tolist()}  dtype={(co_thieu_na == 'HPG').dtype}")
print()
print(f"dtype 'str'    → ô thiếu là {co_thieu_nan[1]!r}")
print(f"  mask (== 'HPG'): {(co_thieu_nan == 'HPG').tolist()}  dtype={(co_thieu_nan == 'HPG').dtype}")

dtype 'string' → ô thiếu là <NA>
  mask (== 'HPG'): [True, <NA>]  dtype=boolean

dtype 'str'    → ô thiếu là nan
  mask (== 'HPG'): [True, False]  dtype=bool


Kiểu `string` cho ra mask `boolean` **có thể chứa `NA`**; kiểu `str` cho ra
mask `bool` thuần. Phần 4 của notebook này chỉ ra vì sao mask chứa `NA` không
lọc được — và đây chính là con đường nó lọt vào code của bạn.

Trong thực tế cột `symbol` của finlens không bao giờ thiếu, nên bạn sẽ không
gặp. Nhưng sau một `merge` không khớp hết thì có.

## 3 · Đo bộ nhớ — chỗ tiết kiệm 40 lần

`.memory_usage(deep=True)` là hàm bạn cần. Không có `deep=True` thì nó chỉ đếm
con trỏ, không đếm nội dung chuỗi — và cho ra một con số sai lệch.

In [12]:
toan_san = client.eod.stock.ohlcv(
    client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist(),
    start=lui_ngay(HOM_NAY, nam=1),
)
print(f"Frame lớn: {len(toan_san):,} dòng × {toan_san.shape[1]} cột")

sym = toan_san["symbol"]
so_sanh = pd.DataFrame(
    {
        "KB": [
            sym.memory_usage(deep=True) / 1024,
            sym.astype(object).memory_usage(deep=True) / 1024,
            sym.astype("category").memory_usage(deep=True) / 1024,
        ]
    },
    index=["str (mặc định 3.0)", "object (mặc định 2.x)", "category"],
).round(1)
so_sanh["so với category"] = (so_sanh["KB"] / so_sanh["KB"].min()).round(1).astype(str) + "×"
so_sanh

Frame lớn: 100,198 dòng × 7 cột


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


,KB,so với category
str (mặc định 3.0),5088.3,23.5×
object (mặc định 2.x),5088.3,23.5×
category,216.4,1.0×


`category` lưu **một bảng mã duy nhất** cộng một mảng số nguyên trỏ vào bảng
đó. Cột `symbol` chỉ có vài trăm giá trị khác nhau lặp lại hàng trăm nghìn lần,
nên đó đúng là hình dạng mà `category` sinh ra để xử lý.

⚠️ Nhưng `category` **không miễn phí**: nối hai frame có tập mã khác nhau sẽ
cho ra `object`, và một số phép chuỗi chậm hơn. Notebook `58` đo cả hai chiều.

In [13]:
print(f"Số giá trị khác nhau: {sym.nunique()} trên {len(sym):,} dòng")
print(f"→ tỷ lệ lặp: mỗi mã xuất hiện trung bình {len(sym) / sym.nunique():.0f} lần")
print()
print("Bộ nhớ toàn frame:")
print((toan_san.memory_usage(deep=True) / 1024**2).round(2).astype(str).add(" MB").to_string())

Số giá trị khác nhau: 405 trên 100,198 dòng
→ tỷ lệ lặp: mỗi mã xuất hiện trung bình 247 lần

Bộ nhớ toàn frame:
Index      0.0 MB
symbol    4.97 MB
date      0.76 MB
open      0.76 MB
high      0.76 MB
low       0.76 MB
close     0.76 MB
volume    0.76 MB


## 4 · `pd.NA` khác `np.nan` — và vì sao điều đó quan trọng

pandas có **hai hệ thống thiếu dữ liệu** sống song song. Biết mình đang ở hệ
nào quyết định kết quả phép so sánh.

In [14]:
kieu_cu = pd.Series([1, None, 3])  # float64, thiếu = np.nan
kieu_moi = pd.Series([1, None, 3], dtype="Int64")  # Int64 nullable, thiếu = pd.NA

print(f"pd.Series([1, None, 3])                 → dtype {str(kieu_cu.dtype):<10} thiếu là {kieu_cu[1]!r}")
print(f"pd.Series([1, None, 3], dtype='Int64')  → dtype {str(kieu_moi.dtype):<10} thiếu là {kieu_moi[1]!r}")

pd.Series([1, None, 3])                 → dtype float64    thiếu là np.float64(nan)
pd.Series([1, None, 3], dtype='Int64')  → dtype Int64      thiếu là <NA>


Chú ý dòng đầu: bạn đưa vào **số nguyên**, nhận về **`float64`**. `np.nan` là
một giá trị dấu phẩy động, nên mảng phải là float mới chứa được nó. Đó là lý do
lịch sử khiến pandas không có số nguyên thiếu dữ liệu — cho tới khi `Int64` viết
hoa ra đời.

In [15]:
print(f"1 == 1.0 nên phép tính vẫn đúng, nhưng kiểu đã đổi:")
print(f"  kieu_cu.sum()  = {kieu_cu.sum()}  ({type(kieu_cu.sum()).__name__})")
print(f"  kieu_moi.sum() = {kieu_moi.sum()}  ({type(kieu_moi.sum()).__name__})")

1 == 1.0 nên phép tính vẫn đúng, nhưng kiểu đã đổi:
  kieu_cu.sum()  = 4.0  (float64)
  kieu_moi.sum() = 4  (int64)


### Logic ba giá trị: `NA > 0` không phải `False`

Đây là khác biệt có hậu quả thật. `np.nan` trong phép so sánh cho ra `False`;
`pd.NA` cho ra **`NA`** — vì "không biết giá trị" thì cũng không biết nó có lớn
hơn 0 hay không.

In [16]:
print("So sánh với np.nan (float64):")
print(f"  {pd.Series([1.0, np.nan]) > 0}".replace("\n", "\n  "))
print("\nSo sánh với pd.NA (Int64):")
print(f"  {pd.Series([1, None], dtype='Int64') > 0}".replace("\n", "\n  "))

So sánh với np.nan (float64):
  0     True
  1    False
  dtype: bool

So sánh với pd.NA (Int64):
  0    True
  1    <NA>
  dtype: boolean


In [17]:
mask_cu = pd.Series([1.0, np.nan, 3.0]) > 2
mask_moi = pd.Series([1, None, 3], dtype="Int64") > 2

print(f"mask từ float64: {mask_cu.tolist()}  dtype={mask_cu.dtype}")
print(f"mask từ Int64  : {mask_moi.tolist()}  dtype={mask_moi.dtype}")
print()
print("Dùng mask để lọc:")
print(f"  float64 → {len(pd.Series([1.0, np.nan, 3.0])[mask_cu])} dòng")
try:
    pd.Series([1, None, 3], dtype="Int64")[mask_moi]
except Exception as e:
    print(f"  Int64   → {type(e).__name__}: {str(e)[:80]}")

mask từ float64: [False, False, True]  dtype=bool
mask từ Int64  : [False, <NA>, True]  dtype=boolean

Dùng mask để lọc:
  float64 → 1 dòng


Boolean mask chứa `NA` **không lọc được** — pandas từ chối đoán hộ bạn. Cách
xử lý là nói rõ ý định bằng `.fillna(False)`:

In [18]:
an_toan = mask_moi.fillna(False)
print(f"mask.fillna(False) → {an_toan.tolist()}, lọc ra {len(pd.Series([1, None, 3], dtype='Int64')[an_toan])} dòng")

mask.fillna(False) → [False, False, True], lọc ra 1 dòng


⚠️ **`finlens` trả về `float64`, không phải `Int64`** — kể cả cột `volume` vốn
là số nguyên về bản chất. Nên trong 17 notebook trước, mọi thứ thiếu dữ liệu đều
là `np.nan` và boolean mask luôn chạy. Bạn chỉ gặp `pd.NA` khi tự ép kiểu, hoặc
khi đọc dữ liệu từ nguồn khác.

In [19]:
print("Kiểu của các cột số trong frame finlens:")
print(gia.select_dtypes("number").dtypes.to_string())
print(f"\nvolume có phải số nguyên về bản chất không? {(gia['volume'] % 1 == 0).all()}")
print(f"nhưng dtype vẫn là: {gia['volume'].dtype}")

Kiểu của các cột số trong frame finlens:
open      float64
high      float64
low       float64
close     float64
volume    float64

volume có phải số nguyên về bản chất không? True
nhưng dtype vẫn là: float64


## 5 · Xem nhanh một frame lạ

Bốn lệnh, theo thứ tự nên gọi khi gặp dữ liệu chưa biết gì.

In [20]:
gia.info()

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   symbol  195 non-null    string        
 1   date    195 non-null    datetime64[ns]
 2   open    195 non-null    float64       
 3   high    195 non-null    float64       
 4   low     195 non-null    float64       
 5   close   195 non-null    float64       
 6   volume  195 non-null    float64       
dtypes: datetime64[ns](1), float64(5), string(1)
memory usage: 10.8 KB


In [21]:
gia.describe().round(2)

,date,open,high,low,close,volume
count,195,195.00,195.00,195.00,195.00,195.00
mean,2026-06-28 04:48:00,51.47,52.05,50.91,51.40,12138613.32
min,2026-05-14 00:00:00,20.15,21.00,20.10,20.35,140100.00
25%,2026-06-05 00:00:00,23.98,24.15,23.72,23.90,4769600.50
50%,2026-06-29 00:00:00,60.79,61.29,60.50,60.79,7772300.00
75%,2026-07-21 00:00:00,70.35,71.05,70.07,70.35,16549300.00
max,2026-08-12 00:00:00,76.70,77.70,75.80,76.64,75910592.00
std,NaN,20.77,21.08,20.55,20.78,10643235.53


`describe()` chỉ lấy cột số. Muốn xem cột chuỗi thì nói rõ:

In [22]:
gia.describe(include=["str"])

,symbol
count,195
unique,3
top,FPT
freq,65


In [23]:
print("Kiểm tra thiếu dữ liệu — luôn làm trước khi tính toán:")
thieu = pd.DataFrame(
    {"số ô thiếu": gia.isna().sum(), "tỷ lệ %": (gia.isna().mean() * 100).round(2)}
)
print(thieu.to_string())

Kiểm tra thiếu dữ liệu — luôn làm trước khi tính toán:
        số ô thiếu  tỷ lệ %
symbol           0      0.0
date             0      0.0
open             0      0.0
high             0      0.0
low              0      0.0
close            0      0.0
volume           0      0.0


## 6 · Ép kiểu — `astype` và `convert_dtypes`

In [24]:
mau = gia.head(200).copy()

ep = mau.astype({"symbol": "category", "volume": "int64"})
print("Sau astype:")
print(ep.dtypes.to_string())
print(f"\nBộ nhớ: {mau.memory_usage(deep=True).sum() / 1024:.1f} KB → {ep.memory_usage(deep=True).sum() / 1024:.1f} KB")

Sau astype:
symbol          category
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64

Bộ nhớ: 19.2 KB → 9.6 KB


⚠️ `astype("int64")` **không chịu được `NaN`**. Đây là chỗ ép kiểu thất bại
giữa chừng trong pipeline:

In [25]:
co_thieu = mau.copy()
co_thieu.loc[co_thieu.index[0], "volume"] = np.nan

try:
    co_thieu.astype({"volume": "int64"})
except Exception as e:
    print(f"astype('int64') với NaN → {type(e).__name__}: {str(e)[:70]}")

print(f"astype('Int64') với NaN → OK, thiếu thành {co_thieu.astype({'volume': 'Int64'})['volume'].iloc[0]!r}")

astype('int64') với NaN → IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or rem
astype('Int64') với NaN → OK, thiếu thành <NA>


`convert_dtypes()` tự chọn kiểu hẹp nhất còn giữ được dữ liệu — tiện khi khám
phá, nhưng đừng dùng trong pipeline vì kết quả phụ thuộc vào **dữ liệu của lần
chạy đó**, và một tháng sau cột có thêm một ô thiếu là kiểu đổi.

In [26]:
tu_dong = mau.convert_dtypes()
print(pd.DataFrame({"trước": mau.dtypes, "sau convert_dtypes": tu_dong.dtypes}).to_string())

                 trước sau convert_dtypes
symbol          string             string
date    datetime64[ns]     datetime64[ns]
open           float64            Float64
high           float64            Float64
low            float64            Float64
close          float64            Float64
volume         float64              Int64


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Xem cấu trúc frame lạ | `df.info()` rồi `df.describe()` |
| Đo bộ nhớ thật | `df.memory_usage(deep=True)` — **luôn có `deep=True`** |
| Đánh số lại nhãn hàng | `df.reset_index(drop=True)` |
| Số nguyên có thiếu dữ liệu | `astype("Int64")` viết hoa |
| Cột lặp lại nhiều | `astype("category")` |

**Bốn điều mang sang notebook sau:**

1. **pandas 3.0 dùng `str` thay `object`** cho chuỗi. Code cũ kiểm
   `df.dtypes == object` để tìm cột chuỗi sẽ không tìm thấy gì nữa. Và có
   **hai** kiểu chuỗi: `str` (thiếu = `np.nan`) là mặc định của pandas, còn
   `string` (thiếu = `pd.NA`) là thứ finlens trả về — khác nhau ở kiểu của
   boolean mask.
2. **`Index` là nhãn, không phải vị trí.** Sau khi lọc, `iloc[0]` và `loc[0]`
   trỏ hai hàng khác nhau — và khi frame chưa lọc thì chúng trùng nhau nên lỗi
   ẩn mình.
3. **`pd.NA` lan qua phép so sánh**, `np.nan` thì không. Boolean mask có `NA`
   không lọc được; phải `.fillna(False)` để nói rõ ý định.
4. `category` cho cột `symbol` tiết kiệm hàng chục lần bộ nhớ, nhưng có cái giá
   riêng — notebook `58` đo cả hai chiều.

---

**Tiếp theo:** [`52_chon_loc_va_cow.ipynb`](52_chon_loc_va_cow.ipynb) — chọn và
lọc dữ liệu, và cái bẫy Copy-on-Write mà pandas 3.0 vừa bật vĩnh viễn.